In [0]:
# Setup da camada Gold
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import *
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

spark.sql("CREATE DATABASE IF NOT EXISTS gold")

DataFrame[]

# **gold.fact_movies_performance**

In [0]:
# Um registro único por filme
# Consolida as métricas financeiras (orçamento, receita, lucro) e de engajamento (popularidade, votos, notas)
# Filtrada apenas para filmes lançados, garantindo que não se duplique ao juntar com as dimensões


# Remove a tabela anterior, caso exista
spark.sql("DROP TABLE IF EXISTS gold.fact_movies_performance")

# Pega os filmes lançados direto da dim_movies
df_filmes_lancados = (
    spark.table("gold.dim_movies")
    .filter(F.col("status_filme") == "Lançado")
    .select("sk_movie_id", "id_filme")
)

# Joins passando apenas "id_filme"
df_fact = (
    df_filmes_lancados
    .join(spark.table("silver.tb_financeiro_filmes"), "id_filme", "left")
    .join(spark.table("silver.tb_metricas_engajamento"), "id_filme", "left")
    .select(
        "sk_movie_id",
        F.col("orcamento_usd").cast("DECIMAL(18,2)"),
        F.col("receita_usd").cast("DECIMAL(18,2)"),
        F.col("lucro_usd").cast("DECIMAL(18,2)"),
        F.col("orcamento_brl").cast("DECIMAL(18,2)"),
        F.col("receita_brl").cast("DECIMAL(18,2)"),
        F.col("lucro_brl").cast("DECIMAL(18,2)"),
        "popularidade",
        "nota_media_tmdb",
        F.col("qtd_votos_tmdb").cast("INT"),
        "nota_media_imdb",
        F.col("qtd_votos_imdb").cast("INT")
    )
)

# Salva a tabela fato em Delta
df_fact.write.format("delta").mode("overwrite").saveAsTable("gold.fact_movies_performance")

display(df_fact.limit(10))


sk_movie_id,orcamento_usd,receita_usd,lucro_usd,orcamento_brl,receita_brl,lucro_brl,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
0,null,null,null,null,null,null,null,null,null,null,null
1,null,null,null,null,null,null,null,null,null,null,null
2,null,null,null,null,null,null,null,null,null,null,null
3,null,null,null,null,null,null,null,null,null,null,null
4,null,null,null,null,null,null,null,null,null,null,null
5,null,null,null,null,null,null,null,null,null,null,null
6,null,null,null,null,null,null,null,null,null,null,null
7,4700000.00,null,null,23358060.00,null,null,1.489,6.75,6,6.2,382
8,null,null,null,null,null,null,0.6,0.0,0,7.7,25
9,6000000.00,null,null,29818800.00,null,null,13.212,6.8,15,null,236


# **gold.dim_movies**

In [0]:
# Gera-se as chaves substitutas (sk_movie_id, sk_person_id, sk_company_id) usando row_number/monotonically_increasing_id
# Garante-se que cada dimensão contenha entidades únicas e deduplicadas


# Remove a tabela anterior, caso exista
spark.sql("DROP TABLE IF EXISTS gold.dim_movies")

# Cria a dimensão de filmes com uma chave substituta
df_dim_movies = spark.table("silver.tb_info_filmes").select(
    F.monotonically_increasing_id().alias("sk_movie_id"),
    F.col("id_filme").cast("STRING").alias("id_filme"),
    "titulo",
    "data_lancamento",
    "ano_lancamento",
    "duracao_minutos",
    "idioma_original",
    "status_filme",
    "sinopse"
)

# Salva a dimensão em Delta
df_dim_movies.write.format("delta").mode("overwrite").saveAsTable("gold.dim_movies")

display(df_dim_movies)

sk_movie_id,id_filme,titulo,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse
0,1000004,Purple Beatz,2022-07-07,2022,86,en,Lançado,"Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry."
1,1000005,Aisha Brown: The First Black Woman Ever,2020-02-14,2020,42,en,Lançado,"No one, and nothing, is off-limits for comedian Aisha Brown, as she takes on her boyfriend’s penis, racism, clinical depression, and Donald Trump, in this hilarious one-hour comedy special from Just For Laughs."
2,1000007,KYLE BROWNRIGG: INTRODUCING LYLE,2022-05-27,2022,36,en,Lançado,"Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle."
3,1000011,Worth Your Weight in Gold,2022-07-14,2022,26,pt,Lançado,"Oscar, a renowned hypnotherapist, uses the last moments in his hotel room to say goodbye to the vertiginous Mercedes and rid old Norberto of his bitterness."
4,1000014,On va manquer !,2018-05-15,2018,0,fr,Lançado,null
5,1000030,58 Hours: The Baby Jessica Story,2021-07-31,2021,0,es,Lançado,null
6,1000054,One Hundred Years and Hope,2022-06-18,2022,107,ja,Lançado,"In a country ruled by the Liberal Democratic Party, running on austerity and neoliberal ambitions, for most of its postwar years, gender and economic inequalities have become increasingly acute in Japan. Takashi Nishihara, a filmmaker who has been following the youth protests in Japan notices that there is one party that seems to be raising issues of gender and economic in the political sphere, the Japanese Communist Party (JCP), a party about to enter its hundredth year and consistently burdened by its historical connotations. Though an outsider of the party, Nishihara gained unprecedented access to the JCP and driven by his interest in the younger party members who find hope in the JCP, the resulting documentary goes beyond party politics and observes the current grassroots leftist movements in Japan. It also becomes witness to the larger and deep-seated patriarchal system that continues to quell momentums of hope."
7,1000058,Homecoming,2023-07-12,2023,110,fr,Lançado,"Kheìdidja, in her forties, works for a wealthy Parisian family who offers her the opportunity to take care of their children for a summer in Corsica. It's an opportunity for her to return with her daughters, Jessica and Farah, to the island they left fifteen years earlier in tragic circumstances."
8,1000059,素敵な選TAXI SPECIAL〜湯けむり連続選択肢〜,2016-04-05,2016,116,ja,Lançado,"Edawakare, the driver of the Time Taxi that allows passengers to return to their life's turning points, visits a hot spring this time around. An array of guests at the inn are fraught with troubles in their life and become passengers of the Time Taxi. And somehow all the clients are actually part of a bigger picture?!"
9,1000073,A Chance To Win,2023-05-03,2023,97,fr,Lançado,"Two villages in the south of France have always been bitter rivals, but when a group of asylum seekers arrive in the community, the life of both villages is shaken up and age-old disagreements escalate. Their antagonism reaches its peak with the annual rugby derby played between the two village teams, but this time, with the new outsiders joining as unexpected recruits, the result of the 100th match will be more unpredictable than ever."


# **gold.dim_genres**

In [0]:
# Remove a tabela anterior, caso exista
spark.sql("DROP TABLE IF EXISTS gold.dim_genres")

# Cria um catálogo único de gêneros
df_dim_genres = (spark.table("silver.tb_generos")
    .select("nome_genero")
    .distinct()
    .select(
        F.monotonically_increasing_id().alias("sk_genre_id"),
        "nome_genero"
    )
)

# Salva a dimensão em Delta
df_dim_genres.write.format("delta").mode("overwrite").saveAsTable("gold.dim_genres")

display(df_dim_genres)

sk_genre_id,nome_genero
0,Mystery
1,Animation
2,Science Fiction
3,Action
4,Family
5,Romance
6,History
7,Comedy
8,Crime
9,Thriller


# **gold.dim_people**

In [0]:
# Remove a tabela anterior, caso exista
spark.sql("DROP TABLE IF EXISTS gold.dim_people")

# Filtra pessoas físicas e deduplica primeiro
df_dim_people = (spark.table("silver.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(
        F.col("nome_entidade").alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa")
    )
    .distinct()
    
    # Gera o ID único para cada pessoa/papel
    .withColumn("sk_person_id", F.monotonically_increasing_id())
    .select("sk_person_id", "nome_pessoa", "tipo_pessoa")
)

# Salva a dimensão em Delta
df_dim_people.write.format("delta").mode("overwrite").saveAsTable("gold.dim_people")

display(df_dim_people.limit(10))


sk_person_id,nome_pessoa,tipo_pessoa
0,Don Cheadle,Ator
1,Hugh Jackman,Ator
2,Anthony Gonzalez,Ator
3,Daniel Kaluuya,Ator
4,Mike Myers,Ator
5,Mark Rylance,Ator
6,Ewan Mcgregor,Ator
7,Tom Hardy,Ator
8,Evangeline Lilly,Ator
9,Randall Park,Ator


# **gold.dim_companies**

In [0]:
# Remove a tabela anterior, caso exista
spark.sql("DROP TABLE IF EXISTS gold.dim_companies")

# Filtra produtoras e pega apenas os nomes distintos
df_dim_companies = (spark.table("silver.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(F.col("nome_entidade").alias("nome_produtora"))
    .distinct()  
    
    # Gera um ID único para cada produtora do catálogo
    .withColumn("sk_company_id", F.monotonically_increasing_id())
    .select("sk_company_id", "nome_produtora")
)

# Salva a dimensão em Delta
df_dim_companies.write.format("delta").mode("overwrite").saveAsTable("gold.dim_companies")

display(df_dim_companies.limit(10))


sk_company_id,nome_produtora
0,Gk Films
1,Regency Enterprises
2,Matt Tolmach Productions
3,Walt Disney Pictures
4,Gunn Films
5,Offspring Entertainment
6,Evergreen Media Group
7,Seven Bucks Productions
8,Huayi Brothers Pictures
9,Ratpac Entertainment


# **gold.dim_reviews**

In [0]:
# Remove a tabela anterior, caso exista
spark.sql("DROP TABLE IF EXISTS gold.dim_reviews")

df_movies = spark.table("gold.dim_movies").select("sk_movie_id", "id_filme")

# Resume as avaliações dos usuários por filme
df_dim_reviews = (spark.table("silver.tb_avaliacoes_usuarios")
    .groupBy("id_filme")
    .agg(
        F.count("*").cast("INT").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).cast("DOUBLE").alias("nota_media_usuarios")
    )
    .join(df_movies, on="id_filme", how="inner")
    
    # Gera a Surrogate Key e mantém as colunas do PDF
    .withColumn("sk_review_id", F.monotonically_increasing_id())
    .select(
        "sk_review_id",
        "sk_movie_id",
        "qtd_avaliacoes_usuarios",
        "nota_media_usuarios"
    )
)

# Salva a dimensão em Delta
df_dim_reviews.write.format("delta").mode("overwrite").saveAsTable("gold.dim_reviews")

display(df_dim_reviews.limit(10))


sk_review_id,sk_movie_id,qtd_avaliacoes_usuarios,nota_media_usuarios
0,0,1,0.1
1,4,1,3.3
2,7,1,6.8
3,16,1,5.4
4,18,1,5.3
5,19,1,4.4
6,22,3,4.75
7,23,1,4.8
8,25,1,6.6
9,32,1,7.5


# **gold.bridge_movie_genre**

In [0]:
# Tabelas-ponte (Bridge Tables):
# Um filme pode ter vários gêneros, vários atores e várias produtoras (relação N para N)
# Usa-se as bridge tables para ligar os filmes a essas dimensões sem duplicar as linhas da tabela fact


# Remove a tabela anterior, caso exista
spark.sql("DROP TABLE IF EXISTS gold.bridge_movie_genre")

# Relaciona cada filme aos seus gêneros
df_bridge_genre = (spark.table("silver.tb_generos")
    .join(
        spark.table("gold.dim_movies").select("sk_movie_id", "id_filme"),
        "id_filme",
        "inner"
    )
    .join(spark.table("gold.dim_genres"),
        "nome_genero",
        "inner"
    )
    .select(
        "sk_movie_id",
        "sk_genre_id"
    )
    .dropDuplicates()
)

# Salva a tabela ponte em Delta
df_bridge_genre.write.format("delta").mode("overwrite").saveAsTable("gold.bridge_movie_genre")

display(df_bridge_genre.limit(10))


sk_movie_id,sk_genre_id
18631,10
18656,3
41616,9
23613,14
20818,16
38678,9
51387,16
27987,15
87627,2
75978,4


# **gold.bridge_movie_person**

In [0]:
# Remove a tabela anterior, caso exista
spark.sql("DROP TABLE IF EXISTS gold.bridge_movie_person")

# Relaciona cada filme às pessoas envolvidas
df_bridge_person = (spark.table("silver.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .join(
        spark.table("gold.dim_movies").select("sk_movie_id", "id_filme"),
        "id_filme",
        "inner"
    )
    # Join com a dim_people para buscar a sk_person_id
    .join(spark.table("gold.dim_people"),
        (F.col("nome_entidade") == F.col("nome_pessoa")) &
        (F.col("tipo_entidade") == F.col("tipo_pessoa")),
        "inner"
    )
    .select(
        "sk_movie_id",
        "sk_person_id"
    )
    .dropDuplicates()
)

# Salva a tabela ponte em Delta
df_bridge_person.write.format("delta").mode("overwrite").saveAsTable("gold.bridge_movie_person")

display(df_bridge_person.limit(10))


sk_movie_id,sk_person_id
19723,209555
22181,209560
59622,4
73184,26373
19740,78836
73578,52431
50386,367701
20429,157413
63898,26380
39576,13


# **gold.bridge_movie_company**

In [0]:
# Remove a tabela anterior, caso exista
spark.sql("DROP TABLE IF EXISTS gold.bridge_movie_company")

# Relaciona cada filme às suas produtoras
df_bridge_company = (spark.table("silver.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade") == "Produtora")
    .join(
        spark.table("gold.dim_movies").select("sk_movie_id", "id_filme"),
        "id_filme",
        "inner"
    )
    # Join com a dim_companies para buscar a sk_company_id
    .join(spark.table("gold.dim_companies"),
        F.col("nome_entidade") == F.col("nome_produtora"),
        "inner"
    )
    .select(
        "sk_movie_id",
        "sk_company_id"
    )
    .dropDuplicates()
)

# Salva a tabela ponte em Delta
df_bridge_company.write.format("delta").mode("overwrite").saveAsTable(
    "gold.bridge_movie_company"
)

display(df_bridge_company.limit(10))


sk_movie_id,sk_company_id
18746,20500
30883,2904
19102,40677
18841,20506
38189,8669
21927,5760
18267,2913
61452,13
62661,23340
90573,29111
